# Notebook 14 — Re-extração corrigida das features do PulseDB

**Motivo.** A extração usada até aqui (Notebooks 07 e 09) recorta cada ciclo cardíaco
**entre dois picos sistólicos consecutivos**:

```python
ini = indices_picos[i]; fim = indices_picos[i+1]
ciclo = ppg_filt[ini:fim]
```

Como o recorte *começa* num pico, o `argmax` do ciclo cai necessariamente numa das bordas.
A função `extrair_features_ciclo` usa esse índice para separar sístole de diástole:

```python
peak_idx = int(np.argmax(seg_norm))
sut = peak_idx / fs ;  dt = (n - peak_idx) / fs
sys_idx = indices[indices <= peak_idx] ;  dia_idx = indices[indices > peak_idx]
```

Com o pico na borda, um dos dois lados fica sempre vazio. Resultado medido em 2.182 ciclos
reais do `VitalDB_CalBased_Test_Subset.mat`:

| Posição de `peak_idx` | Frequência | Efeito |
|---|---|---|
| índice 0 (início) | 56,4 % | `sys_idx` com 1 elemento → `sw = 0` → `w*_ratio = NaN` |
| índice n−1 (fim) | 43,6 % | `dia_idx` vazio → `dw = 0` → `w*_ratio = 0` |
| **interior** | **0,0 %** | — |

**21 das 41 features ficam degeneradas**: `systolic_upstroke_time_s`, `diastolic_time_s`
(que vira cópia literal de `cardiac_period_s`), `sut_dt_ratio` e as 18 colunas
`w*_sys` / `w*_dia` / `w*_ratio`.

**Correção.** Recortar o ciclo **vale-a-vale** (pé-a-pé): o vale é o mínimo entre dois picos
sistólicos consecutivos, de modo que cada ciclo contém exatamente um pico **no interior**.
É a definição fisiológica correta de ciclo cardíaco no PPG — o pé da onda marca o início da
ejeção.

**Escopo.** Este notebook grava em arquivos com sufixo `_v3`, **sem sobrescrever** os
originais. Assim os Notebooks 08, 09 e 11 permanecem consistentes com os dados que
efetivamente usaram, e ficam documentados como baseados na extração antiga.

## 1. Importações e configuração

In [1]:
import os
import socket
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
from scipy.stats import skew, kurtosis

warnings.filterwarnings("ignore")
np.random.seed(42)

plt.rcParams["figure.facecolor"]  = "white"
plt.rcParams["axes.facecolor"]    = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["savefig.dpi"]       = 300

print("Computador:", socket.gethostname())
print("Inicio:", datetime.now().strftime("%d/%m/%Y %H:%M:%S"))

Computador: NtebookLeticia
Inicio: 25/07/2026 13:52:53


In [2]:
# ─── Raiz unica do projeto (mesma usada nos demais notebooks) ───
BASE_IC            = Path(r"C:\IC_PPG")
PASTA_PULSEDB      = (BASE_IC / "data" / "pulsedb_kaggle" / "datasets" /
                      "weinanwangrutgers" / "pulsedb-balanced-training-and-testing" /
                      "versions" / "4")
PASTA_PULSEDB_FEAT = BASE_IC / "data" / "pulsedb_features"
PASTA_FIGS         = BASE_IC / "figures"

ARQ_CALBASED = PASTA_PULSEDB / "VitalDB_CalBased_Test_Subset.mat"
SAIDA_V3     = PASTA_PULSEDB_FEAT / "features_pulsedb_calbased_test_v3.csv"

# ─── Parametros do pipeline (identicos aos NB07/NB09, para isolar a mudanca) ───
FS = 125
FREQ_CORTE_INF, FREQ_CORTE_SUP, ORDEM_FILTRO = 0.5, 8.0, 4
FC_MAXIMA_BPM = 200
DISTANCIA_MINIMA_AMOSTRAS = int((60.0 / FC_MAXIMA_BPM) * FS)
NIVEIS_LARGURA = [10, 25, 33, 50, 66, 75]

print(f"Entrada: {ARQ_CALBASED.name} ({ARQ_CALBASED.stat().st_size/1e6:.0f} MB)")
print(f"Saida:   {SAIDA_V3.name}")
print(f"FS={FS} Hz | filtro {FREQ_CORTE_INF}-{FREQ_CORTE_SUP} Hz ordem {ORDEM_FILTRO}")
print(f"Distancia minima entre picos: {DISTANCIA_MINIMA_AMOSTRAS} amostras")

Entrada: VitalDB_CalBased_Test_Subset.mat (1533 MB)
Saida:   features_pulsedb_calbased_test_v3.csv
FS=125 Hz | filtro 0.5-8.0 Hz ordem 4
Distancia minima entre picos: 37 amostras


## 2. Funções de processamento

In [3]:
def projetar_filtro_butterworth(fs, freq_inf, freq_sup, ordem):
    """Passa-banda Butterworth (mesmo dos NB07/NB09)."""
    nyq = 0.5 * fs
    return butter(ordem, [freq_inf / nyq, freq_sup / nyq], btype="band")


def aplicar_filtro_bidirecional(sinal, b, a):
    """filtfilt: filtragem sem deslocamento de fase."""
    return filtfilt(b, a, sinal)


def detectar_picos_sistolicos(ppg_filtrado, distancia_minima):
    """Picos sistolicos (mesmo criterio dos NB07/NB09)."""
    altura_min = np.mean(ppg_filtrado) + 0.1 * np.std(ppg_filtrado)
    indices, _ = find_peaks(ppg_filtrado, distance=distancia_minima, height=altura_min)
    return indices


def detectar_vales(ppg_filtrado, indices_picos):
    """CORRECAO: pe da onda = minimo entre dois picos sistolicos consecutivos.

    Recortar o ciclo entre vales garante que o pico sistolico fique no INTERIOR
    do recorte, tornando bem definidas a subida sistolica e a fase diastolica.
    Com o recorte antigo (pico-a-pico) o pico caia sempre numa borda e 21 das 41
    features degeneravam.
    """
    return np.array([p + int(np.argmin(ppg_filtrado[p:q]))
                     for p, q in zip(indices_picos[:-1], indices_picos[1:])])


print("Funcoes de filtragem e deteccao prontas (com detectar_vales).")

Funcoes de filtragem e deteccao prontas (com detectar_vales).


### 2.1 Extração das 41 features

A função é **idêntica** à dos Notebooks 07 e 09 — nenhuma linha foi alterada. A correção está
apenas em *como o ciclo é recortado* antes de chamá-la. Isso isola a mudança: qualquer
diferença nos resultados vem da segmentação, não da fórmula das features.

In [4]:
def extrair_features_ciclo(ppg_segmento, fs):
    """Extrai 41 features morfologicas de um unico ciclo cardiaco PPG.

    IDENTICA a dos NB07/NB09. Agora recebe um ciclo recortado VALE-A-VALE,
    de modo que peak_idx cai no interior e sistole/diastole ficam bem definidas.
    """
    n = len(ppg_segmento)
    duracao_s = n / fs

    if n < int(0.33 * fs) or duracao_s > 2.0:
        return None

    seg_min, seg_max = ppg_segmento.min(), ppg_segmento.max()
    seg_range = seg_max - seg_min
    if seg_range < 1e-10:
        return None

    seg_norm = (ppg_segmento - seg_min) / seg_range
    feats = {}

    # ===== TEMPORAIS =====
    feats["cardiac_period_s"] = duracao_s
    feats["heart_rate_bpm"]   = 60.0 / duracao_s

    peak_idx = int(np.argmax(seg_norm))
    sut = peak_idx / fs
    dt  = (n - peak_idx) / fs
    feats["systolic_upstroke_time_s"] = sut
    feats["diastolic_time_s"]         = dt
    feats["sut_dt_ratio"] = sut / dt if dt > 0.02 else np.nan

    # ===== LARGURAS DE PULSO =====
    for pct in NIVEIS_LARGURA:
        above = seg_norm >= (pct / 100.0)
        if above.sum() > 0:
            indices = np.where(above)[0]
            total_w = (indices[-1] - indices[0]) / fs
            sys_idx = indices[indices <= peak_idx]
            dia_idx = indices[indices >  peak_idx]
            sw = (sys_idx[-1] - sys_idx[0]) / fs if len(sys_idx) > 1 else 0.0
            dw = (dia_idx[-1] - dia_idx[0]) / fs if len(dia_idx) > 1 else 0.0
            feats[f"w{pct}_total"] = total_w
            feats[f"w{pct}_sys"]   = sw
            feats[f"w{pct}_dia"]   = dw
            feats[f"w{pct}_ratio"] = dw / sw if sw > 0.005 else np.nan
        else:
            feats[f"w{pct}_total"] = 0.0
            feats[f"w{pct}_sys"]   = 0.0
            feats[f"w{pct}_dia"]   = 0.0
            feats[f"w{pct}_ratio"] = np.nan

    # ===== AMPLITUDE (ciclo BRUTO) =====
    feats["amplitude"]    = float(seg_max - seg_min)
    feats["peak_value"]   = float(seg_max)
    feats["valley_value"] = float(seg_min)
    feats["crest_factor"] = float(seg_max / (abs(seg_min) if abs(seg_min) > 1e-9 else 1e-9))

    # ===== ESTATISTICAS =====
    feats["variance"] = float(np.var(ppg_segmento))
    feats["std"]      = float(np.std(ppg_segmento))
    feats["skewness"] = float(skew(seg_norm))
    feats["kurtosis"] = float(kurtosis(seg_norm, fisher=True))
    feats["auc"]      = float(np.trapz(seg_norm) / fs)

    # ===== HJORTH =====
    activity = float(np.var(ppg_segmento))
    diff1 = np.diff(ppg_segmento)
    diff2 = np.diff(diff1)
    mobility = float(np.sqrt(np.var(diff1) / activity)) if activity > 1e-12 else np.nan
    var_d1 = float(np.var(diff1))
    if var_d1 > 1e-12 and not np.isnan(mobility) and mobility > 1e-12:
        complexity = float(np.sqrt(np.var(diff2) / var_d1) / mobility)
    else:
        complexity = np.nan
    feats["hjorth_activity"]   = activity
    feats["hjorth_mobility"]   = mobility
    feats["hjorth_complexity"] = complexity
    return feats


print("Funcao de extracao pronta (41 features).")

Funcao de extracao pronta (41 features).


## 3. Demonstração do defeito e da correção

Comparação lado a lado nos mesmos segmentos, antes de processar a base inteira.

In [5]:
coef_b, coef_a = projetar_filtro_butterworth(FS, FREQ_CORTE_INF, FREQ_CORTE_SUP, ORDEM_FILTRO)
arq = h5py.File(ARQ_CALBASED, "r")
sinais = arq["Subset"]["Signals"]     # (1250 amostras, 3 canais, 51720 segmentos)
N_SEGMENTOS = sinais.shape[2]
print(f"Signals: {sinais.shape}  ->  {N_SEGMENTOS} segmentos de "
      f"{sinais.shape[0]/FS:.0f} s (canal 1 = PPG)\n")

def diagnostico(modo, passo=20, limite=4000):
    """Onde cai o pico e o que acontece com w50_ratio, no recorte dado."""
    tot = ini = fim = meio = 0
    r_nan = r_zero = r_ok = sut_zero = 0
    for idx in range(0, limite, passo):
        filt = aplicar_filtro_bidirecional(sinais[:, 1, idx], coef_b, coef_a)
        picos = detectar_picos_sistolicos(filt, DISTANCIA_MINIMA_AMOSTRAS)
        if len(picos) < 3:
            continue
        cortes = picos if modo == "pico" else detectar_vales(filt, picos)
        for i in range(len(cortes) - 1):
            ciclo = filt[cortes[i]:cortes[i+1]]
            fe = extrair_features_ciclo(ciclo, FS)
            if fe is None:
                continue
            tot += 1
            n = len(ciclo)
            pk = int(np.argmax((ciclo - ciclo.min()) / (ciclo.max() - ciclo.min())))
            if pk == 0:      ini += 1
            elif pk == n-1:  fim += 1
            else:            meio += 1
            if fe["systolic_upstroke_time_s"] == 0:
                sut_zero += 1
            v = fe["w50_ratio"]
            if np.isnan(v):   r_nan += 1
            elif v == 0.0:    r_zero += 1
            else:             r_ok += 1
    return {"modo": modo, "n_ciclos": tot,
            "pico_inicio_%": 100*ini/tot, "pico_fim_%": 100*fim/tot,
            "pico_interior_%": 100*meio/tot, "sut_zero_%": 100*sut_zero/tot,
            "w50ratio_NaN_%": 100*r_nan/tot, "w50ratio_zero_%": 100*r_zero/tot,
            "w50ratio_util_%": 100*r_ok/tot}

diag = pd.DataFrame([diagnostico("pico"), diagnostico("vale")])
diag["recorte"] = ["ATUAL (pico-a-pico)", "CORRIGIDO (vale-a-vale)"]
print(diag.set_index("recorte").drop(columns="modo").T.to_string(
    float_format=lambda v: f"{v:9.1f}"))
diag.to_csv(PASTA_PULSEDB_FEAT / "nb14_segmentation_diagnosis.csv", index=False)
print("\n[salvo] nb14_segmentation_diagnosis.csv")

Signals: (1250, 3, 51720)  ->  51720 segmentos de 10 s (canal 1 = PPG)



recorte          ATUAL (pico-a-pico)  CORRIGIDO (vale-a-vale)
n_ciclos                      2236.0                   2061.0
pico_inicio_%                   55.3                      0.0
pico_fim_%                      44.7                      0.0
pico_interior_%                  0.0                    100.0
sut_zero_%                      55.3                      0.0
w50ratio_NaN_%                  55.3                      0.0
w50ratio_zero_%                 44.7                      0.0
w50ratio_util_%                  0.0                    100.0

[salvo] nb14_segmentation_diagnosis.csv


## 4. Re-extração completa (51.720 segmentos)

In [6]:
print(f"Iniciando extracao vale-a-vale em {N_SEGMENTOS} segmentos...")
inicio = datetime.now()

# Metadados escalares do subset (uma entrada por segmento)
SBP  = arq["Subset"]["SBP"][0, :]
DBP  = arq["Subset"]["DBP"][0, :]
IDADE  = arq["Subset"]["Age"][0, :]
ALTURA = arq["Subset"]["Height"][0, :]
PESO   = arq["Subset"]["Weight"][0, :]
REF_SUBJ = arq["Subset"]["Subject"][0, :]
REF_GEN  = arq["Subset"]["Gender"][0, :]

def texto(ref):
    """Converte referencia HDF5 do MATLAB em string."""
    return "".join(chr(c[0]) for c in arq[ref][:])

linhas = []
problemas = {"sem_picos": 0, "ciclo_invalido": 0}

for idx in range(N_SEGMENTOS):
    filt = aplicar_filtro_bidirecional(sinais[:, 1, idx], coef_b, coef_a)
    picos = detectar_picos_sistolicos(filt, DISTANCIA_MINIMA_AMOSTRAS)
    if len(picos) < 3:                      # precisa de >=3 picos p/ ter >=2 vales
        problemas["sem_picos"] += 1
        continue
    vales = detectar_vales(filt, picos)     # <<< CORRECAO

    sbp, dbp = float(SBP[idx]), float(DBP[idx])
    subject_id = "v" + texto(REF_SUBJ[idx])
    genero = texto(REF_GEN[idx])
    idade, altura, peso = float(IDADE[idx]), float(ALTURA[idx]), float(PESO[idx])

    for i in range(len(vales) - 1):
        ini_c, fim_c = vales[i], vales[i+1]
        feats = extrair_features_ciclo(filt[ini_c:fim_c], FS)
        if feats is None:
            problemas["ciclo_invalido"] += 1
            continue
        feats.update({
            "record": f"seg_{idx:05d}", "subject": subject_id, "condition": "rest",
            "cycle_idx": i, "peak_sample": int(ini_c), "time_s": float(ini_c / FS),
            "bp_sys_start": sbp, "bp_sys_end": sbp,
            "bp_dia_start": dbp, "bp_dia_end": dbp,
            "bp_sys_mean": sbp, "bp_dia_mean": dbp,
            "age": idade, "gender": genero, "height": altura, "weight": peso,
        })
        linhas.append(feats)

    if (idx + 1) % 5000 == 0:
        seg_s = (datetime.now() - inicio).total_seconds()
        print(f"  {idx+1:>6d}/{N_SEGMENTOS} segmentos | {len(linhas):>7d} ciclos | "
              f"{seg_s/60:.1f} min")

arq.close()
duracao = (datetime.now() - inicio).total_seconds()
print(f"\nExtracao concluida em {duracao/60:.1f} min")
print(f"  ciclos extraidos:      {len(linhas)}")
print(f"  segmentos sem picos:   {problemas['sem_picos']}")
print(f"  ciclos invalidos:      {problemas['ciclo_invalido']}")

Iniciando extracao vale-a-vale em 51720 segmentos...


    5000/51720 segmentos |   51180 ciclos | 1.5 min


   10000/51720 segmentos |  103752 ciclos | 2.8 min


   15000/51720 segmentos |  157518 ciclos | 4.1 min


   20000/51720 segmentos |  209939 ciclos | 5.5 min


   25000/51720 segmentos |  262406 ciclos | 12.1 min


... (2 outputs left out)

   40000/51720 segmentos |  415728 ciclos | 14.8 min


   45000/51720 segmentos |  467395 ciclos | 15.7 min


   50000/51720 segmentos |  518186 ciclos | 16.6 min



Extracao concluida em 17.0 min
  ciclos extraidos:      536375
  segmentos sem picos:   0
  ciclos invalidos:      2315


## 5. DataFrame, imputação e verificação

In [7]:
df_v3 = pd.DataFrame(linhas)
FEATS = [c for c in df_v3.columns if c in [
    "cardiac_period_s","heart_rate_bpm","systolic_upstroke_time_s","diastolic_time_s",
    "sut_dt_ratio"] or c.startswith("w") or c in [
    "amplitude","peak_value","valley_value","crest_factor","variance","std",
    "skewness","kurtosis","auc","hjorth_activity","hjorth_mobility","hjorth_complexity"]]
print(f"{len(df_v3)} ciclos | {df_v3.subject.nunique()} sujeitos | "
      f"{df_v3.record.nunique()} segmentos | {len(FEATS)} features")

nan_antes = df_v3[FEATS].isna().sum()
print(f"\nNaN antes da imputacao: {int(nan_antes.sum())} "
      f"({100*nan_antes.sum()/(len(df_v3)*len(FEATS)):.3f}% das celulas)")
if nan_antes.sum():
    print(nan_antes[nan_antes > 0].to_string())

# Imputacao mediana-por-sujeito -> mediana-global (mesma estrategia dos NB05/07/09)
for col in FEATS:
    if df_v3[col].isna().any():
        df_v3[col] = df_v3.groupby("subject")[col].transform(lambda x: x.fillna(x.median()))
        if df_v3[col].isna().any():
            df_v3[col] = df_v3[col].fillna(df_v3[col].median())
assert df_v3[FEATS].isna().sum().sum() == 0
print("Imputacao concluida.")

536375 ciclos | 1293 sujeitos | 51720 segmentos | 42 features



NaN antes da imputacao: 0 (0.000% das celulas)


Imputacao concluida.


In [8]:
# ─── Verificacao: as 21 features antes degeneradas agora tem conteudo? ───
DEGENERADAS_ANTES = (["systolic_upstroke_time_s", "diastolic_time_s", "sut_dt_ratio"] +
                     [f"w{p}_{s}" for p in NIVEIS_LARGURA for s in ("sys", "dia", "ratio")])
antigo = pd.read_csv(PASTA_PULSEDB_FEAT / "features_pulsedb_calbased_test.csv",
                     usecols=DEGENERADAS_ANTES, nrows=200_000)

print(f"{'feature':<26s}{'distintos ANTES':>17s}{'distintos DEPOIS':>18s}"
      f"{'%zeros ANTES':>14s}{'%zeros DEPOIS':>15s}")
linhas_cmp = []
for c in DEGENERADAS_ANTES:
    a_nu, d_nu = antigo[c].nunique(), df_v3[c].nunique()
    a_z, d_z = 100*(antigo[c] == 0).mean(), 100*(df_v3[c] == 0).mean()
    linhas_cmp.append({"feature": c, "distintos_antes": a_nu, "distintos_depois": d_nu,
                       "pct_zeros_antes": a_z, "pct_zeros_depois": d_z})
    print(f"{c:<26s}{a_nu:>17d}{d_nu:>18d}{a_z:>13.1f}%{d_z:>14.1f}%")

cmp_df = pd.DataFrame(linhas_cmp)
cmp_df.to_csv(PASTA_PULSEDB_FEAT / "nb14_feature_comparison.csv", index=False)

# checagem de duplicatas exatas remanescentes
dups = []
for i, a in enumerate(FEATS):
    for b in FEATS[i+1:]:
        if np.allclose(df_v3[a].values[:200000], df_v3[b].values[:200000], rtol=1e-9, atol=0):
            dups.append((a, b))
print(f"\nDuplicatas exatas remanescentes: {len(dups)}")
for a, b in dups:
    print(f"   {a} == {b}")
print("  (esperado: apenas variance == hjorth_activity, que sao a mesma grandeza por definicao)")
print(f"\n[salvo] nb14_feature_comparison.csv")

feature                     distintos ANTES  distintos DEPOIS  %zeros ANTES  %zeros DEPOIS
systolic_upstroke_time_s                194               143         56.4%           0.0%
diastolic_time_s                        175               193          0.0%           0.0%
sut_dt_ratio                              4              7547        100.0%           0.0%
w10_sys                                 195               136         56.4%           0.0%
w10_dia                                 182               189         43.6%           0.0%
w10_ratio                                 4              5654        100.0%           0.0%
w25_sys                                 195               115         56.4%           0.0%
w25_dia                                 187               167         43.6%           0.0%
w25_ratio                                 4              3418        100.0%           0.0%
w33_sys                                 195               107         56.4%           0.0%

w66_dia                                 196                91         43.6%           0.0%
w66_ratio                                 4               985        100.0%           0.0%
w75_sys                                 207                71         56.4%           0.0%
w75_dia                                 194                81         43.6%           0.0%
w75_ratio                                 4               752        100.0%           0.0%



Duplicatas exatas remanescentes: 1
   variance == hjorth_activity
  (esperado: apenas variance == hjorth_activity, que sao a mesma grandeza por definicao)

[salvo] nb14_feature_comparison.csv


## 6. Salvamento

In [9]:
COLUNAS_ORDEM = FEATS + ["record", "subject", "condition", "cycle_idx", "peak_sample",
                         "time_s", "bp_sys_start", "bp_sys_end", "bp_dia_start",
                         "bp_dia_end", "bp_sys_mean", "bp_dia_mean",
                         "age", "gender", "height", "weight"]
df_v3 = df_v3[COLUNAS_ORDEM]
df_v3.to_csv(SAIDA_V3, index=False)
print(f"[salvo] {SAIDA_V3.name}  ({SAIDA_V3.stat().st_size/1e6:.1f} MB)")
print(f"  {len(df_v3)} ciclos x {len(df_v3.columns)} colunas")
print(f"  {df_v3.subject.nunique()} sujeitos | {df_v3.record.nunique()} segmentos")
print("\nO arquivo original features_pulsedb_calbased_test.csv NAO foi alterado:")
orig = PASTA_PULSEDB_FEAT / "features_pulsedb_calbased_test.csv"
print(f"  {orig.name}: {orig.stat().st_size/1e6:.1f} MB (intacto)")
print(f"\nNotebook 14 concluido em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

[salvo] features_pulsedb_calbased_test_v3.csv  (346.7 MB)
  536375 ciclos x 58 colunas
  1293 sujeitos | 51720 segmentos

O arquivo original features_pulsedb_calbased_test.csv NAO foi alterado:
  features_pulsedb_calbased_test.csv: 360.6 MB (intacto)

Notebook 14 concluido em: 25/07/2026 14:10:46


## 7. Conclusão

A re-extração vale-a-vale corrige as 21 features que estavam degeneradas pela posição do pico
na borda do recorte. O arquivo `features_pulsedb_calbased_test_v3.csv` é a base correta para
análises novas.

**Notebooks baseados na extração antiga** — resultados permanecem válidos *para os dados que
usaram*, mas as features morfológicas de sístole/diástole neles não medem o que o nome indica:

| Notebook | Situação |
|---|---|
| 07 | Extração AAMI — mesmo defeito, não re-extraído aqui |
| 08 | Usa a extração AAMI antiga |
| 09 | Produziu a extração CalBased antiga |
| 11 | Treina no CalBased antigo |
| 13 | **Re-executado sobre a base `_v3`** |